<a href="https://colab.research.google.com/github/nivedita-rmsh/CLED/blob/main/Data_Inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
from collections import Counter

Verifying existence of MAVEN files

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'
for f in os.listdir(maven_dir):
    size = os.path.getsize(os.path.join(maven_dir, f)) / 1024
    print(f"{f:40s} {size:.1f} KB")

test.jsonl                               15380.6 KB
valid.jsonl                              14807.4 KB
train.jsonl                              60456.8 KB
README.md                                2.9 KB


Load and Inspect Structure of JSON files

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(f'{maven_dir}/train.jsonl')

# Look at one document
doc = train[0]
print("Keys in a document:", list(doc.keys()))
print("Title:", doc['title'])
print("Number of sentences:", len(doc['content']))
print("Number of events:", len(doc['events']))

Keys in a document: ['title', 'id', 'content', 'events', 'negative_triggers']
Title: 2006 Pangandaran earthquake and tsunami
Number of sentences: 9
Number of events: 39


Looking into sentences with their event triggers highlighted.

In [ ]:
for doc in train[:3]:
    print(f"\n=== {doc['title']} ===")
    for event in doc['events'][:2]:
        etype = event['type']
        for mention in event['mention'][:1]:
            sid   = mention['sent_id']
            start, end = mention['offset']
            tokens = doc['content'][sid]['tokens']
            trigger = ' '.join(tokens[start:end])
            sentence = ' '.join(tokens)
            print(f"  Event type : {etype}")
            print(f"  Trigger    : '{trigger}'  (tokens {start}–{end})")
            print(f"  Sentence   : {sentence}\n")


=== 2006 Pangandaran earthquake and tsunami ===
  Event type : Know
  Trigger    : 'observed'  (tokens 12–13)
  Sentence   : Several thousand kilometers to the southeast , surges of several meters were observed in northwestern Australia , but in Java the tsunami runups ( height above normal sea level ) were typically and resulted in the deaths of more than 600 people .

  Event type : Warning
  Trigger    : 'warning'  (tokens 27–28)
  Sentence   : since the shock was felt with only moderate intensity well inland , and even less so at the shore , the surge arrived with little or no warning .


=== Battle of Santa Clara (1927) ===
  Event type : Receiving
  Trigger    : 'received'  (tokens 2–3)
  Sentence   : The aircraft received fire from an enemy machine gun and a dive bombing raid ensued , with three bombs being dropped on the Nicaraguan rebels .

  Event type : Motion
  Trigger    : 'dropped'  (tokens 20–21)
  Sentence   : The aircraft received fire from an enemy machine gun and a 

Event Stats

In [ ]:
event_types = Counter()
total_mentions = 0

for doc in train:
    for event in doc['events']:
        event_types[event['type']] += len(event['mention'])
        total_mentions += len(event['mention'])

print(f"Total documents   : {len(train)}")
print(f"Total event types : {len(event_types)}")
print(f"Total mentions    : {total_mentions}")
print(f"\nTop 10 event types:")
for etype, count in event_types.most_common(10):
    print(f"  {etype:30s} {count}")

Total documents   : 2913
Total event types : 168
Total mentions    : 77993

Top 10 event types:
  Catastrophe                    3145
  Attack                         2920
  Hostile_encounter              2856
  Causation                      2728
  Process_start                  2628
  Competition                    2460
  Motion                         2165
  Social_event                   1653
  Killing                        1625
  Conquering                     1432


Convert to BIO format

In [ ]:
def doc_to_bio_examples(doc):
    examples = []
    trigger_map = {}
    for event in doc['events']:
        for mention in event['mention']:
            sid = mention['sent_id']
            trigger_map.setdefault(sid, []).append(
                (mention['offset'][0], mention['offset'][1], event['type'])
            )

    for sid, sent in enumerate(doc['content']):
        tokens = sent['tokens']
        labels = ['O'] * len(tokens)
        for start, end, etype in trigger_map.get(sid, []):
            for i in range(start, end):
                labels[i] = f"B-{etype}" if i == start else f"I-{etype}"
        examples.append({'tokens': tokens, 'labels': labels, 'doc_id': doc['id'], 'sent_id': sid})
    return examples

# Test it
sample = doc_to_bio_examples(train[0])
for ex in sample[:3]:
    pairs = list(zip(ex['tokens'], ex['labels']))
    non_o = [(t, l) for t, l in pairs if l != 'O']
    if non_o:
        print(ex['tokens'])
        print(ex['labels'])
        print()

['The', '2006', 'Pangandaran', 'earthquake', 'and', 'tsunami', 'occurred', 'on', 'July', '17', 'at', 'along', 'a', 'subduction', 'zone', 'off', 'the', 'coast', 'of', 'west', 'and', 'central', 'Java', ',', 'a', 'large', 'and', 'densely', 'populated', 'island', 'in', 'the', 'Indonesian', 'archipelago', '.']
['O', 'O', 'O', 'B-Catastrophe', 'O', 'B-Catastrophe', 'B-Presence', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['The', 'shock', 'had', 'a', 'moment', 'magnitude', 'of', '7.7', 'and', 'a', 'maximum', 'perceived', 'intensity', 'of', 'IV', '(', '``', 'Light', "''", ')', 'in', 'Jakarta', ',', 'the', 'capital', 'and', 'largest', 'city', 'of', 'Indonesia', '.']
['O', 'B-Catastrophe', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Know', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['There', 'were', 'no', 'direct', 'effects', 'of', 'the', 'ear

Sanity Check

In [ ]:
def check_bio_integrity(examples):
    errors = 0
    for ex in examples:
        prev = 'O'
        for i, label in enumerate(ex['labels']):
            if label.startswith('I-'):
                etype = label[2:]
                if prev != f'B-{etype}' and prev != f'I-{etype}':
                    print(f"BIO error at token {i}: '{label}' follows '{prev}'")
                    print(f"  Tokens: {ex['tokens']}")
                    errors += 1
            prev = label
    print(f"\nTotal BIO errors: {errors}")

check_bio_integrity(sample)


Total BIO errors: 0
